# Serif Health Take-Home: Data Integration

Integrating Hospital Price Transparency (HPT) and Transparency in Coverage (TIC) data samples for three NYC hospitals.

In [27]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', 200)

## 1. Data Inspection

In [28]:
hpt_raw = pd.read_csv('hpt_extract_20250213.csv', dtype=str)
tic_raw = pd.read_csv('tic_extract_20250213.csv', dtype=str)

print(f'HPT: {hpt_raw.shape[0]} rows, {hpt_raw.shape[1]} columns')
print(f'TIC: {tic_raw.shape[0]} rows, {tic_raw.shape[1]} columns')

HPT: 2950 rows, 22 columns
TIC: 222 rows, 17 columns


In [29]:
print('=== HPT columns ===')
for col in hpt_raw.columns:
    n_unique = hpt_raw[col].nunique()
    n_null = hpt_raw[col].isna().sum() + (hpt_raw[col] == 'null').sum()
    print(f'  {col:45s}  {n_unique:>5} unique, {n_null:>5} null/missing')

print('\n=== TIC columns ===')
for col in tic_raw.columns:
    n_unique = tic_raw[col].nunique()
    n_null = tic_raw[col].isna().sum() + (tic_raw[col] == 'null').sum()
    print(f'  {col:45s}  {n_unique:>5} unique, {n_null:>5} null/missing')

=== HPT columns ===
  source_file_name                                   3 unique,     0 null/missing
  hospital_id                                        3 unique,     0 null/missing
  hospital_name                                      3 unique,     0 null/missing
  last_updated_on                                    3 unique,     0 null/missing
  hospital_state                                     1 unique,     0 null/missing
  license_number                                     3 unique,     0 null/missing
  payer_name                                        73 unique,     0 null/missing
  plan_name                                        538 unique,     0 null/missing
  code_type                                          3 unique,     0 null/missing
  raw_code                                           4 unique,     0 null/missing
  description                                       20 unique,     0 null/missing
  setting                                            3 unique,     0 null/miss

### Schema Mapping

Before cleaning, here's how the fields in each dataset correspond conceptually:

| Concept | HPT Column | TIC Column |
|---|---|---|
| **Hospital ID** | `source_file_name` (EIN embedded) | `ein` |
| **Payer** | `payer_name` | `payer` |
| **Plan** | `plan_name` | `network_name` |
| **Procedure Code** | `raw_code` + `code_type` | `code` + `code_type` |
| **Negotiated Rate ($)** | `standard_charge_negotiated_dollar` | `rate` (when `negotiation_type` = negotiated) |
| **Rate as %** | `standard_charge_negotiated_percentage` | `rate` (when `negotiation_type` = percentage) |
| **Gross / List Price** | `standard_charge_gross` | *(not available)* |
| **Care Setting** | `setting` (inpatient/outpatient/both) | `billing_class` (institutional/professional) |
| **Place of Service** | *(not available)* | `place_of_service_list` |
| **Rate Methodology** | `standard_charge_methodology` | `negotiation_type` + `arrangement` |
| **Medicare Benchmark** | *(not available)* | `cms_baseline_rate` |
| **Provider Detail** | *(hospital-level only)* | `taxonomy_filtered_npi_list` |

In [30]:
print('=== HPT unique values in key columns ===')
for col in ['hospital_name', 'payer_name', 'code_type', 'raw_code', 'setting',
            'standard_charge_methodology']:
    print(f'\n{col}:')
    print(hpt_raw[col].value_counts().to_string())

print('\n=== TIC unique values in key columns ===')
for col in ['payer', 'code_type', 'code', 'billing_class', 'negotiation_type', 'arrangement']:
    print(f'\n{col}:')
    print(tic_raw[col].value_counts().to_string())

=== HPT unique values in key columns ===

hospital_name:
hospital_name
NYU Langone                  2322
Montefiore Medical Center     380
The Mount Sinai Hospital      248

payer_name:
payer_name
hip                          392
medicare                     336
multiplan                    276
bcbs                         273
healthfirst                  137
aetna                        113
cigna                        102
fidelis                       95
uhc                           89
ghi                           83
somos                         79
MetroPlus                     79
healthplus                    76
metroplus                     74
Emblem                        72
magnacare                     59
Empire                        50
Fidelis                       43
medicaid                      42
HealthFirst                   32
Aetna                         31
centerlight                   24
Multiplan                     23
Magnacare                     21
oxford     

In [31]:
print('=== HPT sample rows ===')
hpt_raw.head(5)

=== HPT sample rows ===


,source_file_name,hospital_id,hospital_name,last_updated_on,hospital_state,license_number,payer_name,plan_name,code_type,raw_code,description,setting,modifiers,standard_charge_gross,standard_charge_discounted_cash,standard_charge_negotiated_dollar,standard_charge_negotiated_percentage,standard_charge_min,standard_charge_max,standard_charge_methodology,additional_payer_notes,additional_generic_notes
0,13-1740114_montefiore-medical-center_standardcharges.csv.gz,62915ae8-8d64-4e2f-b05f-b18edde57a3d,Montefiore Medical Center,2024-07-01,NY,13-1740114,Aetna,Medicare,CPT,99283,EMERGENCY DEPT VISIT LOW MDM,outpatient,NaN,NaN,NaN,323.34,NaN,83.78,1009.22,fee schedule,NaN,NaN
1,13-1740114_montefiore-medical-center_standardcharges.csv.gz,62915ae8-8d64-4e2f-b05f-b18edde57a3d,Montefiore Medical Center,2024-07-01,NY,13-1740114,HealthFirst,Commercial Enrollees,CPT,43239,EGD BIOPSY SINGLE/MULTIPLE,outpatient,NaN,NaN,NaN,1037.65,NaN,165.4,3206.34,fee schedule,NaN,NaN
2,13-1740114_montefiore-medical-center_standardcharges.csv.gz,62915ae8-8d64-4e2f-b05f-b18edde57a3d,Montefiore Medical Center,2024-07-01,NY,13-1740114,Aetna,Commercial,CPT,43239,UPPER GI ENDOSCOPY BIOPSY,outpatient,NaN,NaN,NaN,1246.73,NaN,1246.73,1394.79,fee schedule,NaN,NaN
3,13-1740114_montefiore-medical-center_standardcharges.csv.gz,62915ae8-8d64-4e2f-b05f-b18edde57a3d,Montefiore Medical Center,2024-07-01,NY,13-1740114,Cigna,LocalPlus,CPT,99283,HC EMERGENCY DEPT VISIT LVL 3,outpatient,NaN,3744,2433.6,1797,NaN,225,1797,other,NaN,per visit
4,13-1740114_montefiore-medical-center_standardcharges.csv.gz,62915ae8-8d64-4e2f-b05f-b18edde57a3d,Montefiore Medical Center,2024-07-01,NY,13-1740114,Oscar,Medicare,CPT,43239,EGD BIOPSY SINGLE/MULTIPLE,outpatient,NaN,NaN,NaN,1037.65,NaN,141.77,1815.1,fee schedule,NaN,NaN


In [32]:
print('=== TIC sample rows ===')
tic_raw.head(5)

=== TIC sample rows ===


,payer,network_name,network_id,network_year_month,network_region,code,code_type,ein,taxonomy_filtered_npi_list,modifier_list,billing_class,place_of_service_list,negotiation_type,arrangement,rate,cms_baseline_schedule,cms_baseline_rate
0,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,872,MS-DRG,131740114,"1003990763,1023202793,1063525152,1063606739,1073708392,1...",NaN,institutional,NaN,negotiated,ffs,15902,IPPS,6829.75
1,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,99283,CPT,131624096,"1003255670,1245759711,1487026522,1598095267,1629218284,1...",NaN,professional,11,negotiated,ffs,123.86,PFS_NONFACILITY_1320201,76.89
2,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131740114,"1700348620,1700892056,1922539964,1942685292",NaN,professional,11,negotiated,ffs,993.92,PFS_NONFACILITY_1320202,424.76
3,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,872,MS-DRG,133971298,"1245635200,1437523537,1528013695,1528432622,1538111653,1...",NaN,institutional,NaN,negotiated,ffs,27924.63,IPPS,6829.75
4,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131740114,1346697315,NaN,professional,11,negotiated,ffs,849.63,PFS_NONFACILITY_1320203,391.85


### Granularity Analysis

A single code + payer combination can yield multiple rates in both datasets. Understanding
what drives this multiplicity is essential before attempting to match records across sources.

In [33]:
# TIC: what drives multiple rates for the same code + payer + hospital?
print('=== TIC Granularity ===\n')

tic_group = tic_raw.groupby(['ein', 'payer', 'code', 'code_type'])
print(f'Unique (ein, payer, code, code_type) groups: {tic_group.ngroups}')
print(f'Total rows: {len(tic_raw)}')
print(f'Avg rows per group: {len(tic_raw) / tic_group.ngroups:.1f}\n')

# Pick a concrete example to show what varies within a group
example = tic_raw[(tic_raw['code'] == '99283') & (tic_raw['payer'] == 'unitedhealthcare')
                  & (tic_raw['ein'] == '131624096')]
print(f'Example: CPT 99283 / UHC / Mount Sinai (EIN 131624096) — {len(example)} rows')
print('Differentiating columns:')
for col in ['billing_class', 'taxonomy_filtered_npi_list', 'place_of_service_list',
            'negotiation_type', 'arrangement']:
    n = example[col].nunique()
    if n > 1:
        print(f'  {col}: {n} distinct values')
        for v in example[col].unique():
            display_v = v[:70] + '...' if isinstance(v, str) and len(v) > 70 else v
            print(f'    - {display_v}')

print('\n--- Rate variation within this group ---')
print(example[['billing_class', 'rate', 'negotiation_type',
               'taxonomy_filtered_npi_list']].assign(
    npi_count=example['taxonomy_filtered_npi_list'].str.split(',').str.len(),
    npi_preview=example['taxonomy_filtered_npi_list'].str[:40] + '...'
)[['billing_class', 'rate', 'negotiation_type', 'npi_count', 'npi_preview']].to_string(index=False))

# Check for true duplicates: same NPI list but different rates
from collections import Counter
npi_counts = Counter(example['taxonomy_filtered_npi_list'].values)
dupes = {k: v for k, v in npi_counts.items() if v > 1}
if dupes:
    print(f'\n⚠ {sum(dupes.values())} rows share an NPI list but have different rates:')
    for npi in dupes:
        rows = example[example['taxonomy_filtered_npi_list'] == npi]
        print(f'  NPI {npi}: rates = {rows["rate"].tolist()}')
    print('  All other columns are identical — possible data quality issue or unexposed contract tiers')

=== TIC Granularity ===

Unique (ein, payer, code, code_type) groups: 25
Total rows: 222
Avg rows per group: 8.9

Example: CPT 99283 / UHC / Mount Sinai (EIN 131624096) — 5 rows
Differentiating columns:
  taxonomy_filtered_npi_list: 4 distinct values
    - 1003255670,1245759711,1487026522,1598095267,1629218284,1740647213
    - 1710151550
    - 1043300940,1083773386,1114151396,1255415014,1275531931,1285913715,1326...
    - 1760503387

--- Rate variation within this group ---
billing_class   rate negotiation_type  npi_count                                 npi_preview
 professional 123.86       negotiated          6 1003255670,1245759711,1487026522,1598095...
 professional  55.04       negotiated          1                               1710151550...
 professional  47.86       negotiated          1                               1710151550...
 professional  83.96       negotiated         20 1043300940,1083773386,1114151396,1255415...
 professional  53.46       negotiated          1        

In [34]:
# HPT: what drives multiple rates for the same code + payer + hospital?
print('=== HPT Granularity ===\n')

hpt_group = hpt_raw.groupby(['hospital_name', 'payer_name', 'raw_code', 'code_type'])
print(f'Unique (hospital, payer, code, code_type) groups: {hpt_group.ngroups}')
print(f'Total rows: {len(hpt_raw)}')
print(f'Avg rows per group: {len(hpt_raw) / hpt_group.ngroups:.1f}\n')

# Pick a concrete example
example_h = hpt_raw[(hpt_raw['raw_code'] == '99283') & (hpt_raw['payer_name'] == 'Cigna')
                     & (hpt_raw['hospital_name'] == 'Montefiore Medical Center')]
print(f'Example: CPT 99283 / Cigna / Montefiore — {len(example_h)} rows')
if len(example_h) == 0:
    example_h = hpt_raw[(hpt_raw['raw_code'] == '99283')
                         & (hpt_raw['hospital_name'] == 'NYU Langone')].head(10)
    print(f'  (Fallback: CPT 99283 / NYU Langone — {len(example_h)} rows)')

print('Differentiating columns:')
for col in ['plan_name', 'setting', 'standard_charge_methodology', 'description']:
    n = example_h[col].nunique()
    if n > 1:
        vals = example_h[col].unique()[:6]
        print(f'  {col}: {n} distinct values')
        for v in vals:
            print(f'    - {v}')
        if len(example_h[col].unique()) > 6:
            print(f'    ... and {len(example_h[col].unique()) - 6} more')

print('\n--- Rate variation within this group ---')
print(example_h[['plan_name', 'setting', 'standard_charge_methodology',
                  'standard_charge_negotiated_dollar']].to_string(index=False))

=== HPT Granularity ===

Unique (hospital, payer, code, code_type) groups: 254
Total rows: 2950
Avg rows per group: 11.6

Example: CPT 99283 / Cigna / Montefiore — 3 rows
Differentiating columns:
  plan_name: 2 distinct values
    - LocalPlus
    - Lifesource
  setting: 2 distinct values
    - outpatient
    - both
  standard_charge_methodology: 2 distinct values
    - other
    - fee schedule
  description: 2 distinct values
    - HC EMERGENCY DEPT VISIT LVL 3
    - PR EMERGENCY DEPARTMENT VISIT LOW MDM

--- Rate variation within this group ---
 plan_name    setting standard_charge_methodology standard_charge_negotiated_dollar
 LocalPlus outpatient                       other                              1797
Lifesource       both                fee schedule                            301.08
 LocalPlus       both                fee schedule                            301.08


In [35]:
# Summary: what makes each row unique?
print('=== What drives rate multiplicity ===\n')

print('TIC — a unique rate is defined by:')
print('  (ein, payer, code, code_type) + billing_class + NPI list + place_of_service_list')
print('  • billing_class: professional vs. institutional (different fee schedules)')
print('  • taxonomy_filtered_npi_list: different providers/groups get different rates')
print('  • place_of_service_list: rates can vary by where care is delivered')

print('\nHPT — a unique rate is defined by:')
print('  (hospital, payer, code, code_type) + plan_name + setting + methodology')
print('  • plan_name: sub-tiers within a payer (HMO, PPO, EPO, Medicare Advantage, etc.)')
print('  • setting: inpatient vs. outpatient vs. both')
print('  • standard_charge_methodology: case rate vs. fee schedule vs. percent of charges')
print('  • description: same code can map to different charge descriptions')


## 2. Data Cleaning & Standardization

In [36]:
hpt = hpt_raw.copy()
tic = tic_raw.copy()

### 2.1 EIN Normalization

HPT `license_number` is unreliable as a hospital identifier — Mount Sinai's is a state
license (`330024`) and NYU Langone's is a facility code (`7002053H`). The actual EINs are
embedded in `source_file_name`, which we extract and validate against TIC.

In [37]:
ein_map = {
    '13-1740114_montefiore-medical-center_standardcharges.csv.gz': '131740114',
    '131624096_mount-sinai-hospital_standardcharges.csv.gz':       '131624096',
    '133971298-1801992631_nyu-langone-tisch_standardcharges.csv.gz': '133971298',
}
hpt['ein'] = hpt['source_file_name'].map(ein_map)
assert hpt['ein'].notna().all(), f"Unmapped: {hpt[hpt['ein'].isna()]['source_file_name'].unique()}"

print('EIN overlap with TIC:', set(hpt['ein'].unique()) & set(tic['ein'].unique()))
print(hpt.groupby(['hospital_name', 'ein']).size().reset_index(name='rows').to_string(index=False))

EIN overlap with TIC: {'133971298', '131740114', '131624096'}
            hospital_name       ein  rows
Montefiore Medical Center 131740114   380
              NYU Langone 133971298  2322
 The Mount Sinai Hospital 131624096   248


### 2.2 Payer Normalization

TIC uses three canonical payer names: `unitedhealthcare`, `aetna`, `cigna-corporation`.
HPT uses many variants. We map known synonyms, including Oxford (a UHC subsidiary whose
HPT plan names include "United Oxford").

In [38]:
UHC_NAMES = {'united', 'uhc', 'unitedhealthcare', 'united healthcare', 'united health', 'oxford'}
AETNA_NAMES = {'aetna'}
CIGNA_NAMES = {'cigna', 'cigna-corporation', 'cigna corporation'}

def normalize_payer(name):
    if not isinstance(name, str):
        return None
    key = name.strip().lower()
    if key in UHC_NAMES:
        return 'unitedhealthcare'
    if key in AETNA_NAMES:
        return 'aetna'
    if key in CIGNA_NAMES:
        return 'cigna-corporation'
    return key

hpt['payer'] = hpt['payer_name'].apply(normalize_payer)
tic['payer'] = tic['payer'].str.strip().str.lower()

tic_payers = set(tic['payer'].unique())
print('HPT payer distribution (normalized):')
for p, c in hpt['payer'].value_counts().items():
    marker = ' ** IN TIC' if p in tic_payers else ''
    print(f'  {p:30s} {c:>5}{marker}')

HPT payer distribution (normalized):
  hip                              392
  medicare                         336
  multiplan                        299
  bcbs                             273
  healthfirst                      169
  metroplus                        153
  aetna                            144 ** IN TIC
  unitedhealthcare                 144 ** IN TIC
  fidelis                          138
  cigna-corporation                113 ** IN TIC
  ghi                               83
  magnacare                         80
  somos                             79
  healthplus                        76
  emblem                            72
  empire                            50
  medicaid                          42
  centerlight                       33
  wellcare                          25
  1199                              20
  beacon                            18
  oscar                             16
  affinity                          15
  humana                            

### 2.3 Filter to Common Hospitals & Payers

Keep only rows where both the hospital (by EIN) and the payer exist in both datasets.

In [39]:
common_eins = set(hpt['ein'].unique()) & set(tic['ein'].unique())
common_payers = set(hpt['payer'].unique()) & set(tic['payer'].unique())

print(f'Common EINs:   {common_eins}')
print(f'Common payers: {common_payers}')

hpt_before, tic_before = len(hpt), len(tic)

hpt = hpt[hpt['ein'].isin(common_eins) & hpt['payer'].isin(common_payers)].copy()
tic = tic[tic['ein'].isin(common_eins) & tic['payer'].isin(common_payers)].copy()

print(f'\nHPT: {hpt_before} -> {len(hpt)} rows (kept {len(hpt)/hpt_before:.0%})')
print(f'TIC: {tic_before} -> {len(tic)} rows (kept {len(tic)/tic_before:.0%})')
print(f'\nHPT by hospital × payer:')
print(hpt.groupby(['hospital_name', 'payer']).size().reset_index(name='rows').to_string(index=False))

Common EINs:   {'133971298', '131740114', '131624096'}
Common payers: {'cigna-corporation', 'aetna', 'unitedhealthcare'}

HPT: 2950 -> 401 rows (kept 14%)
TIC: 222 -> 222 rows (kept 100%)

HPT by hospital × payer:
            hospital_name             payer  rows
Montefiore Medical Center             aetna    19
Montefiore Medical Center cigna-corporation     9
Montefiore Medical Center  unitedhealthcare    19
              NYU Langone             aetna   113
              NYU Langone cigna-corporation   102
              NYU Langone  unitedhealthcare   109
 The Mount Sinai Hospital             aetna    12
 The Mount Sinai Hospital cigna-corporation     2
 The Mount Sinai Hospital  unitedhealthcare    16


### 2.4 Code Normalization

Two issues to fix:
1. NYU Langone embeds the code type in `raw_code` (e.g. `"MS-DRG 872"` instead of `"872"`)
2. Both datasets use `"MS-DRG"` with a dash — we strip it to get a clean join key (`"MSDRG"`)

HPT `LOCAL` code types are hospital-internal codes with no TIC counterpart; they're dropped by the common-payer filter or excluded naturally during matching.

In [40]:
hpt['code'] = hpt['raw_code'].str.strip().str.replace(r'^MS-DRG\s+', '', regex=True)
hpt['code_type'] = hpt['code_type'].str.upper().str.strip().str.replace('MS-DRG', 'MSDRG', regex=False)
tic['code_type'] = tic['code_type'].str.upper().str.replace('-', '', regex=False).str.strip()

print('HPT code_type:', hpt['code_type'].value_counts().to_dict())
print('TIC code_type:', tic['code_type'].value_counts().to_dict())
print(f'\nHPT codes: {sorted(hpt["code"].unique())}')
print(f'TIC codes: {sorted(tic["code"].unique())}')
print(f'Code overlap: {set(hpt["code"].unique()) & set(tic["code"].unique())}')

HPT code_type: {'CPT': 279, 'MSDRG': 66, 'LOCAL': 56}
TIC code_type: {'CPT': 203, 'MSDRG': 19}

HPT codes: ['43239', '872', '99283']
TIC codes: ['43239', '872', '99283']
Code overlap: {'99283', '43239', '872'}


### 2.5 Infer Billing Class for HPT

TIC has an explicit `billing_class` (professional vs institutional). HPT does not, but we
can infer it in two reliable cases:
- `code_type = MS-DRG` → institutional (DRGs are always bundled inpatient facility charges)
- Description prefix `PR ` → professional (physician fee), used consistently at Montefiore

The `HC` prefix is **not** a reliable facility indicator — at NYU Langone it's a generic
chargemaster prefix applied inconsistently, with non-HC rows sometimes having higher rates
than HC rows for the same code. All other rows are left as NaN.


In [ ]:
def infer_billing_class(row):
    if row['code_type'] == 'MSDRG':
        return 'institutional'
    if str(row['description']).strip().startswith('PR '):
        return 'professional'
    return np.nan

hpt['billing_class'] = hpt.apply(infer_billing_class, axis=1)

# Drop Psych ED rows — CPT 99283 used for psychiatric ER visits is a different
# clinical context than a general ER visit and not comparable to TIC rates
psych_mask = hpt['description'].str.contains('Psych', case=False, na=False)
print(f'Dropping Psych ED rows: {psych_mask.sum()}')
hpt = hpt[~psych_mask].copy()

print(f'\nHPT billing_class distribution ({len(hpt)} rows):')
print(hpt['billing_class'].value_counts(dropna=False).to_string())
print(f'\nMapping detail:')
for bc, grp in hpt.groupby('billing_class', dropna=False):
    print(f'\n  {bc} ({len(grp)} rows):')
    for desc, count in grp['description'].value_counts().items():
        print(f'    {count:>4}  {desc}')


### 2.6 Rate Columns

Cast rate-related columns to numeric and consolidate into clean column names.

In [42]:
# HPT: cast to numeric and replace original string columns
hpt_rate_cols = {
    'standard_charge_negotiated_dollar': 'negotiated_rate',
    'standard_charge_negotiated_percentage': 'negotiated_pct',
    'standard_charge_gross': 'gross_charge',
    'standard_charge_discounted_cash': 'discounted_cash',
    'standard_charge_min': 'charge_min',
    'standard_charge_max': 'charge_max',
}
for old, new in hpt_rate_cols.items():
    hpt[new] = pd.to_numeric(hpt[old], errors='coerce')
hpt = hpt.drop(columns=list(hpt_rate_cols.keys()))

# TIC: cast to numeric and replace original string columns
tic['negotiated_rate'] = pd.to_numeric(tic.pop('rate'), errors='coerce')
tic['cms_rate'] = pd.to_numeric(tic.pop('cms_baseline_rate'), errors='coerce')
tic = tic.drop(columns=['cms_baseline_schedule'])

print('HPT negotiated_rate: non-null', hpt['negotiated_rate'].notna().sum(), '/', len(hpt))
print('HPT negotiated_pct:  non-null', hpt['negotiated_pct'].notna().sum(), '/', len(hpt))
print('HPT gross_charge:    non-null', hpt['gross_charge'].notna().sum(), '/', len(hpt))
print(f'\nTIC negotiated_rate: non-null {tic["negotiated_rate"].notna().sum()} / {len(tic)}')
print(f'TIC cms_rate:        non-null {tic["cms_rate"].notna().sum()} / {len(tic)}')

HPT negotiated_rate: non-null 373 / 398
HPT negotiated_pct:  non-null 61 / 398
HPT gross_charge:    non-null 264 / 398

TIC negotiated_rate: non-null 222 / 222
TIC cms_rate:        non-null 221 / 222


### 2.7 Fill Missing Rates

The relationship `negotiated_rate = negotiated_pct × gross_charge / 100` holds exactly
(within rounding) for all HPT rows where all three are present. We use this to fill in
whichever value is missing when the other two are available.

In [43]:
has_rate = hpt['negotiated_rate'].notna()
has_pct = hpt['negotiated_pct'].notna()
has_gross = hpt['gross_charge'].notna()

# Verify consistency where all three exist
all3 = has_rate & has_pct & has_gross
computed = hpt['negotiated_pct'] * hpt['gross_charge'] / 100
max_diff = (hpt.loc[all3, 'negotiated_rate'] - computed[all3]).abs().max()
print(f'Consistency check ({all3.sum()} rows with all three): max diff = ${max_diff:.2f}')

# Fill missing negotiated_rate
fill_rate = ~has_rate & has_pct & has_gross
hpt.loc[fill_rate, 'negotiated_rate'] = (
    hpt.loc[fill_rate, 'negotiated_pct'] * hpt.loc[fill_rate, 'gross_charge'] / 100
)

# Fill missing negotiated_pct
fill_pct = has_rate & ~has_pct & has_gross
hpt.loc[fill_pct, 'negotiated_pct'] = (
    hpt.loc[fill_pct, 'negotiated_rate'] / hpt.loc[fill_pct, 'gross_charge'] * 100
)

# Fill missing gross_charge
fill_gross = has_rate & has_pct & ~has_gross
hpt.loc[fill_gross, 'gross_charge'] = (
    hpt.loc[fill_gross, 'negotiated_rate'] / (hpt.loc[fill_gross, 'negotiated_pct'] / 100)
)

print(f'\nFilled: {fill_rate.sum()} rates, {fill_pct.sum()} percentages, {fill_gross.sum()} gross charges')
print(f'HPT negotiated_rate coverage: {has_rate.sum()} -> {hpt["negotiated_rate"].notna().sum()}')

Consistency check (55 rows with all three): max diff = $0.01

Filled: 0 rates, 185 percentages, 5 gross charges
HPT negotiated_rate coverage: 373 -> 373


### 2.8 Drop TIC Percentage Rates

Some TIC rows have `negotiation_type = 'percentage'`, meaning the rate is a percentage of
billed charges (e.g. `rate = 70` means 70% of whatever the hospital bills), not a dollar
amount. Converting these to dollars would require matching with HPT gross charges first.
Since there are only 3 such rows, we drop them to keep all rates in comparable dollar units.

In [44]:
pct_mask = tic['negotiation_type'] == 'percentage'
print(f'TIC percentage-based rates: {pct_mask.sum()} rows (dropping)')
if pct_mask.sum() > 0:
    print(tic.loc[pct_mask, ['payer', 'ein', 'code', 'billing_class', 'negotiated_rate',
                              'negotiation_type', 'cms_rate']].to_string(index=False))

tic = tic[~pct_mask].copy()
print(f'\nTIC after drop: {len(tic)} rows, all dollar-denominated')

TIC percentage-based rates: 3 rows (dropping)
payer       ein  code billing_class  negotiated_rate negotiation_type  cms_rate
aetna 131624096 99283 institutional             70.0       percentage    276.89
aetna 131740114 43239  professional             70.0       percentage    424.76
aetna 131740114 99283  professional             70.0       percentage     78.59

TIC after drop: 219 rows, all dollar-denominated


### 2.9 Flag CMS Outliers

For TIC dollar-amount rows, compute the commercial-to-Medicare ratio. Typical commercial
rates are 1.5–5x Medicare; anything above 20x is flagged for review.

In [45]:
dollar_rows = tic['negotiated_rate'].notna() & (tic['cms_rate'] > 0)
tic.loc[dollar_rows, 'cms_multiple'] = (
    tic.loc[dollar_rows, 'negotiated_rate'] / tic.loc[dollar_rows, 'cms_rate']
)

print(f'CMS multiple stats ({dollar_rows.sum()} rows):')
print(tic.loc[dollar_rows, 'cms_multiple'].describe().to_string())

outliers = dollar_rows & (tic['cms_multiple'] > 20)
print(f'\nOutliers > 20x Medicare: {outliers.sum()} rows')
if outliers.sum() > 0:
    print(tic.loc[outliers, ['payer', 'ein', 'code', 'billing_class',
                              'negotiated_rate', 'cms_rate', 'cms_multiple']].to_string(index=False))

CMS multiple stats (218 rows):
count    218.000000
mean       4.387011
std       11.184437
min        0.590279
25%        0.983458
50%        1.638811
75%        3.679817
max       87.417943

Outliers > 20x Medicare: 8 rows
payer       ein  code billing_class  negotiated_rate  cms_rate  cms_multiple
aetna 133971298 43239  professional          13583.0    155.38     87.417943
aetna 133971298 99283  professional           2490.0     78.59     31.683420
aetna 131740114 43239  professional          11500.0    132.62     86.713919
aetna 131740114 43239  professional           4700.0    132.62     35.439602
aetna 133971298 43239  professional           8125.0    155.38     52.291157
aetna 131740114 43239  professional          11000.0    132.62     82.943749
aetna 131624096 43239  professional           5228.0    151.42     34.526483
aetna 131624096 43239  professional           3703.0    151.42     24.455158


### 2.10 Drop Redundant Columns

Remove raw/original columns that have been superseded by cleaned versions.

In [46]:
# Rate columns already dropped in 2.5; drop remaining superseded metadata columns
hpt = hpt.drop(columns=['source_file_name', 'hospital_id', 'license_number', 'payer_name', 'raw_code'])

print(f'HPT final columns ({len(hpt.columns)}):')
print(f'  {list(hpt.columns)}')
print(f'\nTIC final columns ({len(tic.columns)}):')
print(f'  {list(tic.columns)}')

HPT final columns (21):
  ['hospital_name', 'last_updated_on', 'hospital_state', 'plan_name', 'code_type', 'description', 'setting', 'modifiers', 'standard_charge_methodology', 'additional_payer_notes', 'additional_generic_notes', 'ein', 'payer', 'code', 'billing_class', 'negotiated_rate', 'negotiated_pct', 'gross_charge', 'discounted_cash', 'charge_min', 'charge_max']

TIC final columns (17):
  ['payer', 'network_name', 'network_id', 'network_year_month', 'network_region', 'code', 'code_type', 'ein', 'taxonomy_filtered_npi_list', 'modifier_list', 'billing_class', 'place_of_service_list', 'negotiation_type', 'arrangement', 'negotiated_rate', 'cms_rate', 'cms_multiple']


### 2.11 Final Cleaned Data Summary

In [47]:
print('=== HPT Cleaned ===')
print(f'Shape: {hpt.shape}')
print(f'Hospitals: {hpt["hospital_name"].nunique()} — {sorted(hpt["hospital_name"].unique())}')
print(f'Payers:    {hpt["payer"].nunique()} — {sorted(hpt["payer"].unique())}')
print(f'Codes:     {sorted(hpt["code"].unique())}')
print(f'Code types: {hpt["code_type"].value_counts().to_dict()}')
print(f'Rate coverage: {hpt["negotiated_rate"].notna().sum()} / {len(hpt)}')

print(f'\n=== TIC Cleaned ===')
print(f'Shape: {tic.shape}')
print(f'Payers:    {sorted(tic["payer"].unique())}')
print(f'Codes:     {sorted(tic["code"].unique())}')
print(f'Code types: {tic["code_type"].value_counts().to_dict()}')
print(f'Rate coverage: {tic["negotiated_rate"].notna().sum()} / {len(tic)} (all dollar-denominated)')

=== HPT Cleaned ===
Shape: (398, 21)
Hospitals: 3 — ['Montefiore Medical Center', 'NYU Langone', 'The Mount Sinai Hospital']
Payers:    3 — ['aetna', 'cigna-corporation', 'unitedhealthcare']
Codes:     ['43239', '872', '99283']
Code types: {'CPT': 276, 'MSDRG': 66, 'LOCAL': 56}
Rate coverage: 373 / 398

=== TIC Cleaned ===
Shape: (219, 17)
Payers:    ['aetna', 'cigna-corporation', 'unitedhealthcare']
Codes:     ['43239', '872', '99283']
Code types: {'CPT': 200, 'MSDRG': 19}
Rate coverage: 219 / 219 (all dollar-denominated)


In [48]:
print('=== HPT cleaned sample ===')
hpt.head(5)

=== HPT cleaned sample ===


,hospital_name,last_updated_on,hospital_state,plan_name,code_type,description,setting,modifiers,standard_charge_methodology,additional_payer_notes,additional_generic_notes,ein,payer,code,billing_class,negotiated_rate,negotiated_pct,gross_charge,discounted_cash,charge_min,charge_max
0,Montefiore Medical Center,2024-07-01,NY,Medicare,CPT,EMERGENCY DEPT VISIT LOW MDM,outpatient,NaN,fee schedule,NaN,NaN,131740114,aetna,99283,unclear,323.34,NaN,NaN,NaN,83.78,1009.22
2,Montefiore Medical Center,2024-07-01,NY,Commercial,CPT,UPPER GI ENDOSCOPY BIOPSY,outpatient,NaN,fee schedule,NaN,NaN,131740114,aetna,43239,unclear,1246.73,NaN,NaN,NaN,1246.73,1394.79
3,Montefiore Medical Center,2024-07-01,NY,LocalPlus,CPT,HC EMERGENCY DEPT VISIT LVL 3,outpatient,NaN,other,NaN,per visit,131740114,cigna-corporation,99283,institutional,1797.00,47.996795,3744.0,2433.60,225.00,1797.00
6,Montefiore Medical Center,2024-07-01,NY,HC NY Essential,CPT,PR EDG TRANSORAL BIOPSY SINGLE/MULTIPLE,both,NaN,fee schedule,NaN,NaN,131740114,unitedhealthcare,43239,professional,161.66,4.118726,3925.0,2551.25,146.49,1733.49
7,Montefiore Medical Center,2024-07-01,NY,Medicare,CPT,PR EDG TRANSORAL BIOPSY SINGLE/MULTIPLE,both,NaN,fee schedule,NaN,NaN,131740114,aetna,43239,professional,157.52,4.013248,3925.0,2551.25,146.49,1733.49


In [49]:
print('=== TIC cleaned sample ===')
tic.head(5)

=== TIC cleaned sample ===


,payer,network_name,network_id,network_year_month,network_region,code,code_type,ein,taxonomy_filtered_npi_list,modifier_list,billing_class,place_of_service_list,negotiation_type,arrangement,negotiated_rate,cms_rate,cms_multiple
0,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,872,MSDRG,131740114,"1003990763,1023202793,1063525152,1063606739,1073708392,1...",NaN,institutional,NaN,negotiated,ffs,15902.00,6829.75,2.328343
1,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,99283,CPT,131624096,"1003255670,1245759711,1487026522,1598095267,1629218284,1...",NaN,professional,11,negotiated,ffs,123.86,76.89,1.610873
2,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131740114,"1700348620,1700892056,1922539964,1942685292",NaN,professional,11,negotiated,ffs,993.92,424.76,2.339957
3,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,872,MSDRG,133971298,"1245635200,1437523537,1528013695,1528432622,1538111653,1...",NaN,institutional,NaN,negotiated,ffs,27924.63,6829.75,4.088675
4,unitedhealthcare,choice-plus,592bc118-0dac-4f38-949c-11dc9b3a3879,202501,USA,43239,CPT,131740114,1346697315,NaN,professional,11,negotiated,ffs,849.63,391.85,2.168253


## 3. Worked Examples

Two specific cases from the assignment instructions, illustrating the matching challenges
and how to reason about rate alignment across datasets.

### 3.1 Example 1: CPT 43239 / UHC / Mount Sinai — Rate $6,438

The $6,438 rate appears in both datasets. But both datasets also contain additional rates
for the same hospital/payer/code. What varies?

In [52]:
# The $6,438 match
sinai_ein = '131624096'

hpt_ex1 = hpt[(hpt['code'] == '43239') & (hpt['code_type'] == 'CPT') &
              (hpt['ein'] == sinai_ein) & (hpt['payer'] == 'unitedhealthcare')]
tic_ex1 = tic[(tic['code'] == '43239') & (tic['code_type'] == 'CPT') &
              (tic['ein'] == sinai_ein) & (tic['payer'] == 'unitedhealthcare')]

print(f'HPT rows: {len(hpt_ex1)},  TIC rows: {len(tic_ex1)}')
print(f'Rate $6,438 in HPT: {(hpt_ex1["negotiated_rate"] == 6438).sum()} row(s)')
print(f'Rate $6,438 in TIC: {(tic_ex1["negotiated_rate"] == 6438).sum()} row(s)')

print('\n=== HPT: all Mount Sinai / UHC / CPT 43239 rows ===')
print(hpt_ex1[['plan_name', 'description', 'setting', 'billing_class',
               'negotiated_rate', 'standard_charge_methodology']].to_string(index=False))

print('\n=== TIC: all Mount Sinai / UHC / CPT 43239 rows ===')
print(tic_ex1[['billing_class', 'negotiated_rate', 'negotiation_type']].assign(
    npi_count=tic_ex1['taxonomy_filtered_npi_list'].str.split(',').str.len()
).to_string(index=False))

HPT rows: 6,  TIC rows: 8
Rate $6,438 in HPT: 1 row(s)
Rate $6,438 in TIC: 1 row(s)

=== HPT: all Mount Sinai / UHC / CPT 43239 rows ===
                                            plan_name                                  description    setting billing_class  negotiated_rate standard_charge_methodology
                                            All Payer                   Egd biopsy single/multiple outpatient       unclear          4681.00                   case rate
                  United Oxford Government (Medicare) Egd Transoral Biopsy Single/Multiple - 43239 outpatient       unclear          1048.28                Fee Schedule
                                            All Payer                   Egd biopsy single/multiple outpatient       unclear          6438.00                   case rate
                  United Oxford Government (Medicare)      HC Egd Transoral Biopsy Single/Multiple outpatient institutional          1048.28                Fee Schedule
Medicare Advantage

**What changes when the price varies?**

In **HPT**, the rate varies by:
- `plan_name` — "All Payer" ($6,438) vs. Medicare Advantage plans ($1,048.28) vs. Oxford ($4,681)
- `standard_charge_methodology` — case rate vs. fee schedule
- `description` — different chargemaster entries for the same CPT code

In **TIC**, the rate varies by:
- `billing_class` — institutional ($6,438, $2,670, $4,608, $1,532) vs. professional ($862.79, $514.22, $311.55, $270.93)
- `taxonomy_filtered_npi_list` — different provider groups negotiate different rates

**Can other records be aligned?**

The $6,438 match is high-confidence: same hospital, same payer, same code, identical dollar
amount, and TIC explicitly tags it as `billing_class = institutional`. The HPT side labels
it as a "case rate" — note that "case rate" is a pricing methodology, not a billing class,
but the dollar magnitude ($6,438) is consistent with an institutional/facility charge.

The other HPT rows are different plan tiers: Oxford "All Payer" ($4,681), Oxford/UHC
Medicare Advantage ($1,048.28). These may correspond to a different UHC network than
"choice-plus" (the only network in the TIC extract), but we have no crosswalk to verify.

The TIC professional rates ($270–$862) represent physician fees and could potentially align
with HPT professional charges, but Mount Sinai doesn't use PR-prefixed descriptions for
this code, so those HPT rows have `billing_class = NaN` and we can't confirm alignment.

### 3.2 Example 2: DRG 872 / Mount Sinai / Aetna — Rate $29,259.18

Mount Sinai lists an Aetna HMO/POS/PPO rate of $29,259.18 for DRG 872 (sepsis).
This rate does **not** appear in the Aetna TIC extract. What rates do we see instead?

In [53]:
# HPT: Mount Sinai / Aetna / DRG 872 — all plan tiers
hpt_ex2 = hpt[(hpt['code'] == '872') & (hpt['code_type'] == 'MSDRG') &
              (hpt['ein'] == sinai_ein) & (hpt['payer'] == 'aetna')]
print(f'=== HPT: Mount Sinai / Aetna / DRG 872 — {len(hpt_ex2)} rows ===')
print(hpt_ex2[['plan_name', 'setting', 'billing_class', 'negotiated_rate',
               'standard_charge_methodology']].to_string(index=False))

# TIC: Aetna / Mount Sinai / DRG 872
tic_ex2 = tic[(tic['code'] == '872') & (tic['code_type'] == 'MSDRG') &
              (tic['ein'] == sinai_ein) & (tic['payer'] == 'aetna')]
print(f'\n=== TIC: Mount Sinai / Aetna / DRG 872 — {len(tic_ex2)} rows ===')
print(tic_ex2[['billing_class', 'negotiated_rate', 'negotiation_type', 'network_name']].assign(
    npi_count=tic_ex2['taxonomy_filtered_npi_list'].str.split(',').str.len()
).to_string(index=False))

print(f'\nHPT rate $29,259.18 in TIC: {(tic_ex2["negotiated_rate"] == 29259.18).any()}')
print(f'TIC rates: {sorted(tic_ex2["negotiated_rate"].unique())}')
print(f'HPT rates: {sorted(hpt_ex2["negotiated_rate"].dropna().unique())}')

=== HPT: Mount Sinai / Aetna / DRG 872 — 4 rows ===
                               plan_name   setting billing_class  negotiated_rate standard_charge_methodology
                             HMO/POS/PPO inpatient institutional         29259.18                   case rate
                                Medicare inpatient institutional         13708.20                   case rate
Blackstone, Student Health, Savings Plus inpatient institutional         24577.49                   case rate
                Signature Administrators inpatient institutional         36573.98                   case rate

=== TIC: Mount Sinai / Aetna / DRG 872 — 6 rows ===
billing_class  negotiated_rate negotiation_type               network_name  npi_count
institutional          17966.0       negotiated open-access-managed-choice          2
institutional           8757.0       negotiated open-access-managed-choice          3
institutional          13365.0       negotiated open-access-managed-choice          2
i

**Why doesn't $29,259.18 appear in TIC?**

The HPT rate of $29,259.18 is for "HMO/POS/PPO" — a broad plan grouping. The TIC extract
contains only Aetna's "open-access-managed-choice" network, which may not correspond to
Mount Sinai's HMO/POS/PPO grouping. The TIC rates ($8,757–$17,966) are lower and vary by
NPI list (different provider groups).

**How should we handle this?**

This is a structural mismatch: the hospital publishes one rate for a broad plan tier, while
the payer publishes multiple NPI-specific rates under a specific network that may not map
to that plan tier. The options are:

1. **Report as unmatched**: Note that the HPT rate has no exact TIC counterpart in this
   network extract. This is the most honest approach.
2. **Range comparison**: The HPT rate ($29,259.18) exceeds all TIC rates for this group
   ($8,757–$17,966). This discrepancy could indicate different contract terms, a different
   network, or that the TIC extract is incomplete.
3. **Closest-rate alignment**: Match to the nearest TIC rate ($17,966) with a low confidence
   score, flagging the 63% rate difference.

### 3.3 Structural Observations

**Why do DRG rates behave differently than CPT rates when matching?**

DRG 872 is a bundled inpatient payment — a single lump sum covering the entire hospital
stay (room, nursing, labs, meds). These rates are:
- High-dollar ($8,000–$50,000+), making small percentage differences into large dollar gaps
- Always institutional/facility — there's no professional component in a DRG
- Negotiated at the hospital level with fewer variants per plan (no HC/PR split)
- More sensitive to plan tier: the spread between Aetna's cheapest ($13,708 Medicare) and
  most expensive ($36,574 Signature Administrators) plan for the same DRG is nearly 3:1

CPT codes like 43239 and 99283 are individual service charges that:
- Split into facility and professional components (different billing classes)
- Have many more rate variants due to NPI-level negotiation on the TIC side
- Are lower-dollar, so rate differences are smaller in absolute terms
- Are more likely to find exact matches because both datasets report at finer granularity

This means DRG matching will structurally have fewer exact matches and wider rate gaps.

**Should plan type (PPO, HMO, EPO) be a matching key or a confidence score component?**

It belongs in the **confidence score**, not the matching key. Reasons:
- HPT and TIC use fundamentally different naming: HPT has plan names like
  "unitedhealthcareppo1160" while TIC has network names like "choice-plus" — there's no
  clean crosswalk between the two
- A single TIC network often covers multiple HPT plan types
- Making plan type a hard key would discard valid matches where the plan-to-network mapping
  is unknown (which is most cases)
- As a score component, plan type mismatches reduce confidence without eliminating the match
  entirely — this is more appropriate given the data quality

**When matched records have materially different rates, what are legitimate reasons?**

1. **Different plan tiers**: HPT "HMO/POS/PPO" vs. TIC's specific network. The hospital
   may publish a blended or highest-tier rate while TIC reports the specific network rate.
2. **Different providers (NPIs)**: TIC rates vary by which doctor performs the service.
   HPT reports hospital-level rates that may average or represent a specific tier.
3. **Timing**: HPT may reflect a different contract effective date than TIC. Rate
   renegotiations happen periodically.
4. **Lesser-of clause**: HPT may show a contractual rate that exceeds the gross charge,
   but the actual payment is capped. TIC may reflect the capped amount.
5. **Methodology**: A "case rate" in HPT vs. a "fee schedule" rate in TIC may produce
   different dollar amounts for the same service depending on the pricing basis.
6. **Data errors**: Either dataset may contain stale, duplicated, or incorrectly mapped rates.

**Implications for downstream use**: Matched records with rate disagreements should not
be discarded — they are still informative. The rate delta itself is a valuable signal:
small deltas suggest consistent contract data, while large deltas indicate either different
contract terms or data quality issues that warrant investigation.

## 4. Matching & Scoring

A two-step approach: first generate candidate pairs via deterministic blocking on hard keys,
then score each pair on a 0–100 scale using billing class alignment, rate proximity, and
setting consistency.

### 4.1 Blocking

Generate all candidate pairs by joining on the four hard keys that are unambiguous across
both datasets: `ein`, `code`, `code_type`, and `payer`. This produces a many-to-many
cross of HPT and TIC rows within each block.

In [ ]:
block_keys = ['ein', 'code', 'code_type', 'payer']

hpt_block = hpt.reset_index().rename(columns={'index': 'hpt_idx'})
tic_block = tic.reset_index().rename(columns={'index': 'tic_idx'})

candidates = hpt_block.merge(tic_block, on=block_keys, how='inner', suffixes=('_hpt', '_tic'))

n_hpt_matched = candidates['hpt_idx'].nunique()
n_hpt_unmatched = len(hpt) - n_hpt_matched

print(f'Candidate pairs: {len(candidates)}')
print(f'HPT rows with ≥1 candidate: {n_hpt_matched} / {len(hpt)}')
print(f'HPT rows with no candidate: {n_hpt_unmatched}')
print(f'Avg TIC candidates per HPT row: {len(candidates) / n_hpt_matched:.1f}')

### 4.2 Scoring

Each candidate pair receives a score from 0 to 100, composed of three weighted components:

| Component | Max Points | Signal |
|---|---|---|
| **Rate proximity** | 50 | Strongest signal — identical dollar amounts are strong evidence of the same contract |
| **Billing class** | 30 | Structural alignment — professional↔professional or institutional↔institutional |
| **Setting consistency** | 20 | Weaker signal — HPT `setting` loosely relates to TIC `billing_class` |

Rate proximity is the dominant component because, within a block (same hospital, code,
payer), matching dollar amounts are the most concrete evidence that two rows describe
the same negotiated arrangement.

In [ ]:
# --- Component 1: Billing class alignment (0–30 points) ---
bc_hpt = candidates['billing_class_hpt']
bc_tic = candidates['billing_class_tic']
candidates['score_billing'] = np.where(
    bc_hpt.isna(), 10,                          # HPT unknown — partial credit
    np.where(bc_hpt == bc_tic, 30,              # exact match
             0)                                  # known mismatch
)

# --- Component 2: Rate proximity (0–50 points) ---
rate_h = candidates['negotiated_rate_hpt'].fillna(0)
rate_t = candidates['negotiated_rate_tic'].fillna(0)
both_valid = (rate_h > 0) & (rate_t > 0)
max_rate = np.maximum(rate_h, rate_t)
pct_diff = np.where(max_rate > 0, np.abs(rate_h - rate_t) / max_rate, 1.0)

candidates['score_rate'] = np.where(
    ~both_valid, 0,
    np.where(pct_diff < 0.001, 50,              # exact match (<0.1%)
    np.where(pct_diff < 0.05, 40,               # within 5%
    np.where(pct_diff < 0.20, 25,               # within 20%
    np.where(pct_diff < 0.50, 10,               # within 50%
             0))))                               # >50% off
)

# --- Component 3: Setting consistency (0–20 points) ---
setting = candidates['setting'].fillna('').str.lower()
bc_t = candidates['billing_class_tic'].fillna('').str.lower()
candidates['score_setting'] = np.where(
    (setting == 'inpatient') & (bc_t == 'institutional'), 20,
    np.where((setting == 'outpatient') & (bc_t == 'professional'), 20,
    np.where((setting == 'outpatient') & (bc_t == 'institutional'), 15,
    np.where(setting == 'both', 10,
             5)))
)

candidates['match_score'] = (
    candidates['score_billing'] + candidates['score_rate'] + candidates['score_setting']
)

print('Score component stats:')
for col in ['score_billing', 'score_rate', 'score_setting', 'match_score']:
    s = candidates[col]
    print(f'  {col:20s}  mean={s.mean():5.1f}  median={s.median():5.1f}  min={s.min()}  max={s.max()}')

### 4.3 Match Classification

Classify each candidate pair into confidence tiers, then select the best TIC match for
each HPT row (highest score). Also identify unmatched HPT rows (those with no blocking
candidate at all, typically due to LOCAL code types or plan tiers outside the TIC network).

In [ ]:
candidates['match_tier'] = pd.cut(
    candidates['match_score'],
    bins=[-1, 29, 49, 69, 100],
    labels=['unlikely', 'low', 'medium', 'high']
)

print('=== All candidate pairs by tier ===')
print(candidates['match_tier'].value_counts().sort_index().to_string())

# Best match per HPT row (highest score)
best = candidates.loc[candidates.groupby('hpt_idx')['match_score'].idxmax()]

print(f'\n=== Best match per HPT row ({len(best)} rows) ===')
print(best['match_tier'].value_counts().sort_index().to_string())

# Rate delta for best matches
best['rate_delta'] = best['negotiated_rate_hpt'] - best['negotiated_rate_tic']
best['rate_delta_pct'] = np.where(
    best['negotiated_rate_tic'] > 0,
    best['rate_delta'] / best['negotiated_rate_tic'] * 100, np.nan
)

print(f'\n=== Rate delta for high-confidence matches ===')
high = best[best['match_tier'] == 'high']
print(f'Count: {len(high)}')
print(f'Mean delta: ${high["rate_delta"].mean():.2f} ({high["rate_delta_pct"].mean():.1f}%)')
print(f'Median delta: ${high["rate_delta"].median():.2f} ({high["rate_delta_pct"].median():.1f}%)')
print(f'Exact rate matches (delta < $1): {(high["rate_delta"].abs() < 1).sum()}')

In [ ]:
# Detailed view of high-confidence matches
print('=== High-confidence matches (sample) ===')
show_cols = ['hospital_name', 'code', 'payer', 'billing_class_hpt', 'billing_class_tic',
             'negotiated_rate_hpt', 'negotiated_rate_tic', 'rate_delta',
             'score_billing', 'score_rate', 'score_setting', 'match_score']
print(high[show_cols].head(15).to_string(index=False))

print(f'\n=== High-confidence matches by hospital ===')
print(high.groupby('hospital_name').size().to_string())

print(f'\n=== High-confidence matches by code ===')
print(high.groupby('code').size().to_string())

### 4.4 Unmatched HPT Rows

HPT rows that produced no candidate pairs in the blocking step. These are rows where the
(ein, code, code_type, payer) combination doesn't exist in TIC — typically LOCAL code types
or codes not covered in the TIC extract.

In [54]:
tic['network_name'].unique()

array(['choice-plus', 'open-access-managed-choice', 'national-oap'],
      dtype=object)

In [ ]:
matched_hpt_idxs = set(candidates['hpt_idx'].unique())
all_hpt_idxs = set(range(len(hpt)))
unmatched_idxs = all_hpt_idxs - matched_hpt_idxs

if unmatched_idxs:
    unmatched = hpt.iloc[list(unmatched_idxs)]
    print(f'Unmatched HPT rows: {len(unmatched)} / {len(hpt)}')
    print(f'\nBy code_type:')
    print(unmatched['code_type'].value_counts().to_string())
    print(f'\nBy hospital:')
    print(unmatched['hospital_name'].value_counts().to_string())
else:
    print('All HPT rows have at least one TIC candidate')

print(f'\n=== Summary ===')
print(f'HPT rows:        {len(hpt)}')
print(f'  Matched:       {n_hpt_matched} ({n_hpt_matched/len(hpt):.0%})')
print(f'  Unmatched:     {n_hpt_unmatched} ({n_hpt_unmatched/len(hpt):.0%})')
print(f'  High conf:     {(best["match_tier"] == "high").sum()}')
print(f'  Medium conf:   {(best["match_tier"] == "medium").sum()}')
print(f'  Low conf:      {(best["match_tier"] == "low").sum()}')
print(f'  Unlikely:      {(best["match_tier"] == "unlikely").sum()}')